# SpaceX Falcon 9 - Predictive Analysis (Classification)

Train and evaluate several classification models to predict whether the Falcon 9 first stage will land successfully.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

## Helper function to plot the confusion matrix

In [ ]:
def plot_confusion_matrix(y, y_predict):
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax)
    ax.set_xlabel("Predicted labels")
    ax.set_ylabel("True labels")
    ax.set_title("Confusion Matrix")
    ax.xaxis.set_ticklabels(["did not land", "landed"])
    ax.yaxis.set_ticklabels(["did not land", "landed"])
    plt.show()

## Load the data

In [ ]:
data = pd.read_csv("dataset_part_2.csv")
X = pd.read_csv("dataset_part_3.csv")

Y = data["Class"].to_numpy()

## Standardize the feature data

In [ ]:
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)

## Split into training and test sets

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

print("Test set count:", Y_test.shape[0])

## Logistic Regression

In [ ]:
parameters = {"C": [0.01, 0.1, 1], "penalty": ["l2"], "solver": ["lbfgs"]}
lr = LogisticRegression()

logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)

print("tuned hyperparameters :(best parameters) ", logreg_cv.best_params_)
print("accuracy :", logreg_cv.best_score_)

In [ ]:
logreg_accuracy = logreg_cv.score(X_test, Y_test)
logreg_accuracy

In [ ]:
yhat = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## Support Vector Machine

In [ ]:
parameters = {"kernel": ("linear", "rbf", "poly", "sigmoid"),
              "C": np.logspace(-3, 3, 5),
              "gamma": np.logspace(-3, 3, 5)}
svm = SVC()

svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)

print("tuned hyperparameters :(best parameters) ", svm_cv.best_params_)
print("accuracy :", svm_cv.best_score_)

In [ ]:
svm_accuracy = svm_cv.score(X_test, Y_test)
svm_accuracy

In [ ]:
yhat = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## Decision Tree

In [ ]:
parameters = {"criterion": ["gini", "entropy"],
     "splitter": ["best", "random"],
     "max_depth": [2*n for n in range(1,10)],
     "max_features": ["auto", "sqrt"],
     "min_samples_leaf": [1, 2, 4],
     "min_samples_split": [2, 5, 10]}

tree = DecisionTreeClassifier()

tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)

print("tuned hyperparameters :(best parameters) ", tree_cv.best_params_)
print("accuracy :", tree_cv.best_score_)

In [ ]:
tree_accuracy = tree_cv.score(X_test, Y_test)
tree_accuracy

In [ ]:
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## K Nearest Neighbors

In [ ]:
parameters = {"n_neighbors": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
              "p": [1, 2]}

KNN = KNeighborsClassifier()

knn_cv = GridSearchCV(KNN, parameters, cv=10)
knn_cv.fit(X_train, Y_train)

print("tuned hyperparameters :(best parameters) ", knn_cv.best_params_)
print("accuracy :", knn_cv.best_score_)

In [ ]:
knn_accuracy = knn_cv.score(X_test, Y_test)
knn_accuracy

In [ ]:
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## Compare all models and find the best-performing one

In [ ]:
report = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM", "Decision Tree", "KNN"],
    "Accuracy": [logreg_accuracy, svm_accuracy, tree_accuracy, knn_accuracy]
})
report.sort_values("Accuracy", ascending=False)